# Intraday features: coverage, distributions and correlations (Phase 9, step 3)

Describes the features added by `add_intraday_features` (`src/cryptosense/intraday.py`) on 15-minute BTC/USDT data. Existing data only; no download. This notebook is descriptive: it defines no setup and makes no trading decision.

**Steps**
1. Coverage: number and share of NaN values for each new column, overall and by year.
2. Distributions for 2017–2023 and for 2024 onward.
3. The same summary for the chosen window (hour_utc 14 or 15) versus all other hours, 2024 onward.
4. Correlation matrix of the numeric features, 2024 onward (table and heatmap).
5. Typical 1-hour move grouped by New York hour and by UTC hour, 2024 onward (volatility only).

**Rules for this notebook (fixed before any result was computed)**
- No average or directional statistic of future returns is computed: no mean forward return, no share of up moves, and no relationship between any feature and future price direction.
- `ret_1h`, `ret_4h` and `ret_24h` are summarized by their count, standard deviation, and the mean and quantiles of their absolute value (|ret|) only. On rows at 14:00–15:45 UTC they cover 13:15–16:00 UTC, which contains the 1-hour holds chosen at Gate 1 (notebook 11), so their signed mean and quantiles are not reported.
- `rv_4h`, `rv_24h` and `rel_volume_1h` are numeric features and get the full summary: count, mean, standard deviation and the 1%, 25%, 50%, 75% and 99% quantiles.
- `hour_utc`, `hour_ny`, `prev_day_trend` and `prev_day_vol_regime` are labels. The hours are used only for grouping; the two daily labels are summarized by their category shares (among rows that have a label).
- Year and sample are those of the candle's open time (UTC). 2017–2023 = before 2024-01-01; 2024 onward = on or after 2024-01-01.
- Step 5 uses non-overlapping 1-hour log returns starting on the hour, built open to open from 15-minute candles and dropped if an endpoint candle is missing, as in notebook 11. Only their standard deviation is reported. Each period's `hour_ny` and `hour_utc` are the feature values of the 15-minute candle at the period start.

## Step 0: Imports and settings

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

from cryptosense.data import load_processed, timeframe_to_ms
from cryptosense.features import add_features
from cryptosense.regimes import add_regimes
from cryptosense.intraday import add_intraday_features

DATA_DIR = Path.cwd().parent / "data"
SYMBOL = "BTC/USDT"
SPLIT = pd.Timestamp("2024-01-01", tz="UTC")
CHOSEN_HOURS = [14, 15]
HOUR_MS = timeframe_to_ms("1h")

RETURN_FEATURES = ["ret_1h", "ret_4h", "ret_24h"]
NUMERIC_FEATURES = ["rv_4h", "rv_24h", "rel_volume_1h"]
LABEL_FEATURES = ["prev_day_trend", "prev_day_vol_regime"]
QUANTILES = [0.01, 0.25, 0.50, 0.75, 0.99]

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 100)

## Step 1: Load data and add intraday features

In [2]:
m15 = load_processed(SYMBOL, DATA_DIR, timeframe="15m")
daily = add_regimes(add_features(load_processed(SYMBOL, DATA_DIR)))
feats = add_intraday_features(m15, daily)
NEW_COLUMNS = [c for c in feats.columns if c not in m15.columns]

print(f"15m:   {len(m15):,} candles, {m15.index[0]} -> {m15.index[-1]}")
print(f"daily: {len(daily):,} days, {daily.index[0].date()} -> {daily.index[-1].date()}")
print("new columns:", NEW_COLUMNS)

15m:   321,296 candles, 2017-08-17 04:00:00+00:00 -> 2026-10-21 21:00:00+00:00
daily: 3,352 days, 2017-08-17 -> 2026-10-20
new columns: ['ret_1h', 'ret_4h', 'ret_24h', 'rv_4h', 'rv_24h', 'rel_volume_1h', 'hour_utc', 'hour_ny', 'prev_day_trend', 'prev_day_vol_regime']


## Step 2: Coverage — NaN values per new column, overall and by year

Rows are candles (2017 starts 2017-08-17 04:00; 2026 ends 2026-10-21 21:00).

In [3]:
year = feats.index.year
nan_counts = feats[NEW_COLUMNS].isna().groupby(year).sum()
nan_counts.loc["overall"] = feats[NEW_COLUMNS].isna().sum()
rows = feats.groupby(year).size()
rows.loc["overall"] = len(feats)
nan_counts.insert(0, "candles", rows)
nan_counts.index.name = "year"

nan_shares = nan_counts[NEW_COLUMNS].div(nan_counts["candles"], axis=0)

print("Number of NaN values:")
print(nan_counts.to_string())
print("\nShare of NaN values:")
print(nan_shares.to_string(float_format=lambda v: f"{v:.4f}"))

Number of NaN values:
         candles  ret_1h  ret_4h  ret_24h  rv_4h  rv_24h  rel_volume_1h  hour_utc  hour_ny  prev_day_trend  prev_day_vol_regime
year                                                                                                                           
2017       13103      14      38      129     64     384           2723         0        0            9557                13103
2018       34778      30      94      223    144     840           4529         0        0               0                23588
2019       34923      24      69      117     96     576           2619         0        0               0                    0
2020       35053      32      76       83    128     768           1862         0        0               0                    0
2021       34975      24      61       65     96     576           1902         0        0               0                    0
2022       35040       0       0        0      0       0              0         0 

## Step 3: Summary helpers

- Return features: count of non-NaN values, standard deviation, and the mean and quantiles of |ret|.
- Numeric features: count, mean, standard deviation and quantiles.
- Label features: share of each category among rows with a label, with the number of labelled rows.

In [4]:
def return_summary(frame):
    """Count, std, and mean and quantiles of the absolute value for each return feature."""
    rows = {}
    for col in RETURN_FEATURES:
        x = frame[col].dropna()
        a = x.abs()
        rows[col] = {"count": len(x), "std": x.std(), "mean_abs": a.mean(),
                     **{f"abs_q{int(q * 100):02d}": a.quantile(q) for q in QUANTILES}}
    return pd.DataFrame(rows).T


def numeric_summary(frame):
    """Count, mean, std and quantiles for each numeric feature."""
    rows = {}
    for col in NUMERIC_FEATURES:
        x = frame[col].dropna()
        rows[col] = {"count": len(x), "mean": x.mean(), "std": x.std(),
                     **{f"q{int(q * 100):02d}": x.quantile(q) for q in QUANTILES}}
    return pd.DataFrame(rows).T


def label_shares(frame):
    """Share of each category among labelled rows, for each label feature."""
    rows = []
    for col in LABEL_FEATURES:
        x = frame[col].dropna()
        for value, share in x.value_counts(normalize=True).sort_index().items():
            rows.append({"label": col, "category": value, "share": share, "labelled_rows": len(x)})
    return pd.DataFrame(rows).set_index(["label", "category"])


def side_by_side(summary, samples):
    """Apply a summary to each named sample and join the results with the sample as the outer level."""
    return pd.concat({name: summary(frame) for name, frame in samples.items()})


def show(table, int_cols=("count", "labelled_rows")):
    print(table.to_string(formatters={
        col: (lambda v: f"{v:,.0f}") if col in int_cols else (lambda v: f"{v:.6f}")
        for col in table.columns
    }))

## Step 4: Distributions — 2017–2023 and 2024 onward

In [5]:
samples = {
    "2017–2023": feats[feats.index < SPLIT],
    "2024 onward": feats[feats.index >= SPLIT],
}

print("Return features (std of the signed value; mean and quantiles of |ret|):")
show(side_by_side(return_summary, samples))
print("\nNumeric features:")
show(side_by_side(numeric_summary, samples))
print("\nLabel features (share among labelled rows):")
show(side_by_side(label_shares, samples))

Return features (std of the signed value; mean and quantiles of |ret|):
                      count      std mean_abs  abs_q01  abs_q25  abs_q50  abs_q75  abs_q99
2017–2023   ret_1h  222,779 0.008546 0.004854 0.000039 0.001123 0.002685 0.005797 0.033117
            ret_4h  222,564 0.016151 0.009703 0.000080 0.002170 0.005329 0.012119 0.061248
            ret_24h 222,285 0.039245 0.025895 0.000224 0.006358 0.016245 0.035073 0.136692
2024 onward ret_1h   98,389 0.005027 0.003216 0.000034 0.000915 0.002072 0.004080 0.018761
            ret_4h   98,389 0.009897 0.006520 0.000066 0.001796 0.004182 0.008395 0.036024
            ret_24h  98,389 0.023983 0.017135 0.000199 0.005170 0.012088 0.023665 0.076901

Numeric features:
                            count     mean      std      q01      q25      q50      q75      q99
2017–2023   rv_4h         222,363 0.013081 0.011750 0.001953 0.006192 0.009816 0.016034 0.059165
            rv_24h        219,667 0.034436 0.025757 0.006997 0.018994 0.027898

## Step 5: The chosen window — hour_utc 14 or 15 versus all other hours, 2024 onward

In [6]:
recent = samples["2024 onward"]
in_window = recent["hour_utc"].isin(CHOSEN_HOURS)
window_samples = {
    "hours 14–15 UTC": recent[in_window],
    "other hours": recent[~in_window],
}

print("Return features (std of the signed value; mean and quantiles of |ret|):")
show(side_by_side(return_summary, window_samples))
print("\nNumeric features:")
show(side_by_side(numeric_summary, window_samples))
print("\nLabel features (share among labelled rows):")
show(side_by_side(label_shares, window_samples))

Return features (std of the signed value; mean and quantiles of |ret|):
                         count      std mean_abs  abs_q01  abs_q25  abs_q50  abs_q75  abs_q99
hours 14–15 UTC ret_1h   8,200 0.007424 0.005010 0.000052 0.001333 0.003220 0.006817 0.025951
                ret_4h   8,200 0.011268 0.007933 0.000094 0.002284 0.005415 0.010940 0.036823
                ret_24h  8,200 0.023115 0.017005 0.000243 0.005548 0.012564 0.023427 0.071268
other hours     ret_1h  90,189 0.004750 0.003053 0.000033 0.000888 0.002008 0.003896 0.017687
                ret_4h  90,189 0.009763 0.006391 0.000064 0.001761 0.004081 0.008175 0.035964
                ret_24h 90,189 0.024060 0.017147 0.000197 0.005134 0.012043 0.023684 0.077442

Numeric features:
                               count     mean      std      q01      q25      q50      q75      q99
hours 14–15 UTC rv_4h          8,200 0.010122 0.005970 0.001851 0.006024 0.009013 0.012869 0.029488
                rv_24h         8,200 0.022451 0.010

## Step 6: Correlations between numeric features, 2024 onward

Pearson correlations between the six numeric features on the same row, using every pair of non-NaN values. The hours and the daily labels are not included.

In [7]:
corr_cols = RETURN_FEATURES + NUMERIC_FEATURES
corr = recent[corr_cols].corr()
print(f"rows with all six values: {int(recent[corr_cols].notna().all(axis=1).sum()):,} of {len(recent):,}")
print(corr.to_string(float_format=lambda v: f"{v:.3f}"))

rows with all six values: 98,389 of 98,389
               ret_1h  ret_4h  ret_24h  rv_4h  rv_24h  rel_volume_1h
ret_1h          1.000   0.495    0.195 -0.004   0.009         -0.049
ret_4h          0.495   1.000    0.401 -0.053   0.010         -0.093
ret_24h         0.195   0.401    1.000 -0.140  -0.088         -0.100
rv_4h          -0.004  -0.053   -0.140  1.000   0.675          0.536
rv_24h          0.009   0.010   -0.088  0.675   1.000          0.340
rel_volume_1h  -0.049  -0.093   -0.100  0.536   0.340          1.000


In [8]:
fig = px.imshow(
    corr, x=corr_cols, y=corr_cols, zmin=-1, zmax=1, text_auto=".2f", aspect="equal",
    color_continuous_scale=[[0.0, "#184f95"], [0.5, "#f0efec"], [1.0, "#e34948"]],
    title="BTC/USDT intraday features: correlation matrix, 2024 onward",
)
fig.update_traces(xgap=2, ygap=2, hovertemplate="%{y} vs %{x}: %{z:.3f}<extra></extra>")
fig.update_layout(height=520, width=620, plot_bgcolor="white",
                  coloraxis_colorbar=dict(title="r"), xaxis_title=None, yaxis_title=None)
fig.show()

## Step 7: Typical 1-hour move by New York hour and by UTC hour, 2024 onward

Volatility only: the standard deviation (%) of non-overlapping 1-hour log returns starting on the hour, with the number of periods. Grouped by the period start's `hour_ny` and, separately, by its `hour_utc`.

In [9]:
first_ts, last_ts = int(m15["timestamp"].iloc[0]), int(m15["timestamp"].iloc[-1])
open_15m = m15.set_index("timestamp")["open"]
starts = np.arange(-(-first_ts // HOUR_MS) * HOUR_MS, last_ts - HOUR_MS + 1, HOUR_MS)
log_return = np.log(open_15m.reindex(starts + HOUR_MS).to_numpy() / open_15m.reindex(starts).to_numpy())

periods = pd.DataFrame({"log_return": log_return}, index=pd.to_datetime(starts, unit="ms", utc=True))
print(f"1-hour periods on the grid: {len(periods):,} | dropped (missing candle): {int(periods['log_return'].isna().sum()):,}")
periods = periods.dropna()
periods = periods[periods.index >= SPLIT]
# Hours from the features of the 15-minute candle at each period start (that candle exists for every kept period).
periods = periods.join(feats[["hour_ny", "hour_utc"]].set_axis(feats.index.as_unit(periods.index.unit)))
print(f"1-hour periods, 2024 onward: {len(periods):,}")

def move_by(column):
    g = periods.groupby(column)["log_return"]
    return pd.DataFrame({"n": g.size(), "typical_move_%": 100 * g.std()})

by_clock = pd.concat({"hour_ny": move_by("hour_ny"), "hour_utc": move_by("hour_utc")}, axis=1)
by_clock.index.name = "hour"
by_clock.loc["all hours"] = [len(periods), 100 * periods["log_return"].std()] * 2
print(by_clock.to_string(formatters={
    col: (lambda v: f"{v:,.0f}") if col[1] == "n" else (lambda v: f"{v:.3f}")
    for col in by_clock.columns
}))

1-hour periods on the grid: 80,465 | dropped (missing candle): 173
1-hour periods, 2024 onward: 24,597
          hour_ny                hour_utc               
                n typical_move_%        n typical_move_%
hour                                                    
0           1,025          0.372    1,025          0.494
1           1,027          0.420    1,025          0.558
2           1,022          0.403    1,025          0.476
3           1,025          0.428    1,025          0.403
4           1,025          0.442    1,025          0.373
5           1,025          0.358    1,025          0.373
6           1,025          0.383    1,025          0.440
7           1,025          0.383    1,025          0.416
8           1,025          0.489    1,025          0.450
9           1,025          0.713    1,025          0.381
10          1,025          0.817    1,025          0.370
11          1,025          0.644    1,025          0.373
12          1,025          0.557    1,025 

## Results

**Data.** 321,296 15-minute candles (2017-08-17 04:00 → 2026-10-21 21:00 UTC); 98,389 candles from 2024 onward. Feature values are in log units (not %), except the typical 1-hour move in step 5 (%).

### Coverage (NaN count / share)

| Column | Overall | 2017 | 2018 | 2019 | 2020 | 2021 | 2022 | 2023 | 2024–2026 |
|---|---|---|---|---|---|---|---|---|---|
| ret_1h | 128 / 0.0004 | 14 | 30 | 24 | 32 | 24 | 0 | 4 | 0 |
| ret_4h | 343 / 0.0011 | 38 | 94 | 69 | 76 | 61 | 0 | 5 | 0 |
| ret_24h | 622 / 0.0019 | 129 | 223 | 117 | 83 | 65 | 0 | 5 | 0 |
| rv_4h | 544 / 0.0017 | 64 | 144 | 96 | 128 | 96 | 0 | 16 | 0 |
| rv_24h | 3,240 / 0.0101 | 384 | 840 | 576 | 768 | 576 | 0 | 96 | 0 |
| rel_volume_1h | 13,798 / 0.0429 | 2,723 (0.2078) | 4,529 (0.1302) | 2,619 (0.0750) | 1,862 (0.0531) | 1,902 (0.0544) | 0 | 163 (0.0047) | 0 |
| hour_utc, hour_ny | 0 / 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| prev_day_trend | 9,557 / 0.0297 | 9,557 (0.7294) | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| prev_day_vol_regime | 36,691 / 0.1142 | 13,103 (1.0000) | 23,588 (0.6782) | 0 | 0 | 0 | 0 | 0 | 0 |

Per-year shares for the other cells are in the Step 2 table (all ≤ 0.0293).

### Distributions: return features (std of the signed value; mean and quantiles of |ret|)

| Sample | Feature | Count | Std | Mean abs | abs q01 | abs q25 | abs q50 | abs q75 | abs q99 |
|---|---|---|---|---|---|---|---|---|---|
| 2017–2023 | ret_1h | 222,779 | 0.008546 | 0.004854 | 0.000039 | 0.001123 | 0.002685 | 0.005797 | 0.033117 |
| 2017–2023 | ret_4h | 222,564 | 0.016151 | 0.009703 | 0.000080 | 0.002170 | 0.005329 | 0.012119 | 0.061248 |
| 2017–2023 | ret_24h | 222,285 | 0.039245 | 0.025895 | 0.000224 | 0.006358 | 0.016245 | 0.035073 | 0.136692 |
| 2024 onward | ret_1h | 98,389 | 0.005027 | 0.003216 | 0.000034 | 0.000915 | 0.002072 | 0.004080 | 0.018761 |
| 2024 onward | ret_4h | 98,389 | 0.009897 | 0.006520 | 0.000066 | 0.001796 | 0.004182 | 0.008395 | 0.036024 |
| 2024 onward | ret_24h | 98,389 | 0.023983 | 0.017135 | 0.000199 | 0.005170 | 0.012088 | 0.023665 | 0.076901 |

### Distributions: numeric features

| Sample | Feature | Count | Mean | Std | q01 | q25 | q50 | q75 | q99 |
|---|---|---|---|---|---|---|---|---|---|
| 2017–2023 | rv_4h | 222,363 | 0.013081 | 0.011750 | 0.001953 | 0.006192 | 0.009816 | 0.016034 | 0.059165 |
| 2017–2023 | rv_24h | 219,667 | 0.034436 | 0.025757 | 0.006997 | 0.018994 | 0.027898 | 0.041202 | 0.138089 |
| 2017–2023 | rel_volume_1h | 209,109 | 1.068294 | 0.969920 | 0.197354 | 0.576486 | 0.827392 | 1.235230 | 4.768786 |
| 2024 onward | rv_4h | 98,389 | 0.008484 | 0.005575 | 0.001776 | 0.004901 | 0.007096 | 0.010379 | 0.028452 |
| 2024 onward | rv_24h | 98,389 | 0.022407 | 0.010781 | 0.005872 | 0.015401 | 0.020608 | 0.026609 | 0.060655 |
| 2024 onward | rel_volume_1h | 98,389 | 1.050884 | 1.137666 | 0.152669 | 0.496678 | 0.781448 | 1.230260 | 5.302104 |

### Distributions: label shares (among labelled rows)

| Sample | prev_day_trend down / range / up | prev_day_vol_regime high / low | Labelled rows (trend; vol) |
|---|---|---|---|
| 2017–2023 | 0.3264 / 0.2955 / 0.3781 | 0.4477 / 0.5523 | 213,350; 186,216 |
| 2024 onward | 0.3220 / 0.2643 / 0.4137 | 0.4752 / 0.5248 | 98,389; 98,389 |

### Chosen window, 2024 onward: hours 14–15 UTC (8,200 rows) vs other hours (90,189 rows)

| Feature | Statistic | Hours 14–15 | Other hours |
|---|---|---|---|
| ret_1h | std / mean abs / abs q50 / abs q99 | 0.007424 / 0.005010 / 0.003220 / 0.025951 | 0.004750 / 0.003053 / 0.002008 / 0.017687 |
| ret_4h | std / mean abs / abs q50 / abs q99 | 0.011268 / 0.007933 / 0.005415 / 0.036823 | 0.009763 / 0.006391 / 0.004081 / 0.035964 |
| ret_24h | std / mean abs / abs q50 / abs q99 | 0.023115 / 0.017005 / 0.012564 / 0.071268 | 0.024060 / 0.017147 / 0.012043 / 0.077442 |
| rv_4h | mean / std / q50 / q99 | 0.010122 / 0.005970 / 0.009013 / 0.029488 | 0.008335 / 0.005513 / 0.006954 / 0.028368 |
| rv_24h | mean / std / q50 / q99 | 0.022451 / 0.010687 / 0.020570 / 0.060917 | 0.022402 / 0.010789 / 0.020613 / 0.060622 |
| rel_volume_1h | mean / std / q50 / q99 | 1.035491 / 0.847392 / 0.867601 / 4.196934 | 1.052283 / 1.160456 / 0.775330 / 5.423745 |
| prev_day_trend | down / range / up | 0.3220 / 0.2644 / 0.4137 | 0.3220 / 0.2643 / 0.4137 |
| prev_day_vol_regime | high / low | 0.4751 / 0.5249 | 0.4752 / 0.5248 |

All quantiles (q01, q25, q75) are in the Step 5 tables.

### Correlations, 2024 onward (98,389 rows, all six values present)

| | ret_1h | ret_4h | ret_24h | rv_4h | rv_24h | rel_volume_1h |
|---|---|---|---|---|---|---|
| ret_1h | 1.000 | 0.495 | 0.195 | −0.004 | 0.009 | −0.049 |
| ret_4h | 0.495 | 1.000 | 0.401 | −0.053 | 0.010 | −0.093 |
| ret_24h | 0.195 | 0.401 | 1.000 | −0.140 | −0.088 | −0.100 |
| rv_4h | −0.004 | −0.053 | −0.140 | 1.000 | 0.675 | 0.536 |
| rv_24h | 0.009 | 0.010 | −0.088 | 0.675 | 1.000 | 0.340 |
| rel_volume_1h | −0.049 | −0.093 | −0.100 | 0.536 | 0.340 | 1.000 |

### Typical 1-hour move (%), 2024 onward, by New York hour and by UTC hour

24,597 periods. n per New York hour: 1,025, except 01:00 (1,027), 02:00 (1,022), 17:00 and 18:00 (1,024). n per UTC hour: 1,024–1,025.

| Hour | NY move | UTC move |
|---|---|---|
| 00 | 0.372 | 0.494 |
| 01 | 0.420 | 0.558 |
| 02 | 0.403 | 0.476 |
| 03 | 0.428 | 0.403 |
| 04 | 0.442 | 0.373 |
| 05 | 0.358 | 0.373 |
| 06 | 0.383 | 0.440 |
| 07 | 0.383 | 0.416 |
| 08 | 0.489 | 0.450 |
| 09 | 0.713 | 0.381 |
| 10 | 0.817 | 0.370 |
| 11 | 0.644 | 0.373 |
| 12 | 0.557 | 0.446 |
| 13 | 0.550 | 0.630 |
| 14 | 0.541 | 0.775 |
| 15 | 0.512 | 0.733 |
| 16 | 0.492 | 0.548 |
| 17 | 0.455 | 0.589 |
| 18 | 0.477 | 0.538 |
| 19 | 0.452 | 0.542 |
| 20 | 0.522 | 0.494 |
| 21 | 0.510 | 0.472 |
| 22 | 0.445 | 0.490 |
| 23 | 0.400 | 0.404 |
| All hours | 0.502 | 0.502 |

Highest hour: 10:00 New York (0.817); 14:00 UTC (0.775).

## Decision (Gate 2)

**Decision:** The intraday features are accepted: missing values occur only in warm-up periods and early data gaps (none from 2024 onward), and no pair of features has a correlation above 0.8 (highest: rv_4h and rv_24h, 0.675). The trading window is redefined in New York time as 09:00–10:59 (hour_ny 9 and 10), the two hours around the US market open, replacing 14:00–15:59 UTC. The New York definition follows the open through daylight saving time; its busiest hour (10:00 New York) had the lowest 2024+ cost ratio found (0.367). This decision uses volatility only, not price direction.